# Capstone — Finding Refresh Opportunities in Anonymized Search Content

This notebook is the executable companion to the deployed paper. It uses the bundled starter release and keeps claims bounded to the verified local evaluation.


## 1. Question

Can a ranked model help an editor find likely declining content for refresh review before inspecting the full inventory? The decision output is a review priority, not an automatic publishing action.


In [2]:
from pathlib import Path
import json
import pandas as pd

start = Path.cwd().resolve()
search_roots = [start, *start.parents]

model_path = None
for root in search_roots:
    matches = list(root.glob("**/outputs/model_results.json"))
    if matches:
        model_path = matches[0]
        break

if model_path is None:
    raise FileNotFoundError(
        "model_results.json was not found. Run `python scripts/run_all.py` "
        "from the repository root first."
    )

ROOT = model_path.parent.parent
results = json.loads(model_path.read_text(encoding="utf-8"))
summary = json.loads(
    (ROOT / "outputs" / "summary.json").read_text(encoding="utf-8")
)

print(f"Rows scored: {results['input_rows']:,}")
print(f"Positive rate: {results['target_positive_rate']:.1%}")
print(f"Best model: {results['best_model']['name']}")


Rows scored: 30,000
Positive rate: 54.2%
Best model: random_forest


## 2. Data

The analysis uses `data/raw/content_refresh_anonymized.csv`: 30,000 rows, 44 source columns, 32 pseudonymized clients, trailing-90-day activity totals, and two 30-day comparison windows. The gated 79M-row warehouse release is not used in this final evaluation. Label-derived trend fields, pseudonymous IDs as features, provider/model identifiers, titles, URLs, domains, and raw queries are excluded.


In [3]:
raw = pd.read_csv(ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv')
print(raw.shape)
print(raw['client_id'].nunique(), 'pseudonymized clients')
print(raw['trend_direction'].value_counts(dropna=False).to_dict())


(30000, 44)
32 pseudonymized clients
{'down': 16262, 'stable': 5962, 'up': 4388, 'new': 2236, 'flat': 1152}


## 3. Methodology

The baseline is `days_since_last_update >= 180` and `impressions_90d >= 500`. Candidates are trained with seed 42 using the repository pipeline, which compares logistic regression, a shallow decision tree, and a class-balanced random forest. Validation is a deterministic 20% client holdout (27,675 train rows and 2,325 test rows); trend fields and identifiers are not model features.


In [4]:
print('Split:', results['split_strategy'])
print('Train rows:', results['train_rows'])
print('Test rows:', results['test_rows'])
print('Feature count:', results['feature_count'])
print('Baseline rule: stale content with at least 500 impressions')


Split: client_holdout
Train rows: 27675
Test rows: 2325
Feature count: 52
Baseline rule: stale content with at least 500 impressions


## 4. Results (vs baseline)

The table below reads the committed model receipt, so the notebook reports the same split as the paper. Precision@50 is the operational metric because the first review batch is small.


In [5]:
rows = []
for name, metrics in results['models'].items():
    rows.append({'method': name, 'roc_auc': metrics['roc_auc'], 'average_precision': metrics['average_precision'], 'precision_at_50': metrics['precision_at_50'], 'recall': metrics['recall'], 'f1': metrics['f1']})
baseline = results['baseline']
rows.append({'method': 'baseline_rules', 'roc_auc': baseline['baseline_roc_auc'], 'average_precision': baseline['baseline_average_precision'], 'precision_at_50': baseline['baseline_precision_at_50'], 'recall': None, 'f1': None})
pd.DataFrame(rows).round(3)


,method,roc_auc,average_precision,precision_at_50,recall,f1
0,decision_tree,0.742,0.575,0.62,0.716,0.634
1,logistic_regression,0.700,0.522,0.40,0.567,0.566
2,random_forest,0.747,0.610,0.68,0.741,0.638
3,baseline_rules,0.627,0.468,0.24,NaN,NaN


## 5. Limitations

This is retrospective decision support, not a causal experiment. The decline label is a directional proxy, the sample is one anonymized release, and the score estimates review priority rather than expected lift from an edit. A future prospective test should log interventions and comparable follow-up windows.


## 6. Ranked recommendations

1. Review high-confidence visible pages first.
2. Treat low CTR as a diagnostic candidate, not an automatic rewrite.
3. Separate engagement review from search review.
4. Monitor low-confidence items instead of spending equal effort everywhere.
5. Log outcomes to support a future prospective evaluation.


## 7. Artifacts the paper embeds

The pipeline writes `outputs/charts/`, `outputs/model_results.json`, `outputs/summary.json`, and `outputs/model_report.md`. The deployed page embeds the headline comparison, feature interpretation, and queue mix. The reproducibility command is `pip install -r requirements.txt` followed by `python scripts/run_all.py`.


## Acknowledgments & data credit

Built on the [FlyRank ML Internship dataset](https://flyrank.ai).

## Self-check

- [x] Abstract and all required paper sections are present.
- [x] Claims are bounded to the bundled anonymized starter release.
- [x] No client names, private queries, titles, URLs, or domains are exposed.
- [x] The deployed URL is recorded in `submission/paper_url.txt`.


## 8. Five-minute showcase demo outline

**0:00–0:45 — Question.** FlyRank editors cannot manually inspect every existing page, so can a transparent ranked queue identify likely declining content for refresh review before the full inventory is inspected?

**0:45–2:00 — Method.** Use 30,000 anonymized content items from 32 pseudonymized clients, exclude label-derived fields and identifiers as features, compare the stale-and-visible rule with logistic regression, a decision tree, and a class-balanced random forest, and evaluate on a deterministic client holdout.

**2:00–3:00 — One chart.** Show the `Precision@50` comparison: rule baseline 0.240 versus random forest 0.680 on the same holdout. Explain that this measures the quality of the first 50 review slots, alongside the 54.2% positive-class base rate.

**3:00–4:00 — One honest result.** The random forest improved the measured review-priority ranking, reaching ROC AUC 0.747, but this is retrospective decision support: it does not show that refreshing a page causes more traffic or predict the lift from a specific edit.

**4:00–5:00 — Recommendation.** Start with high-confidence, visible pages; inspect intent, evidence freshness, and the proposed action; use low-confidence items for monitoring; and log interventions plus comparable follow-up windows before claiming operational impact.


## Shareable cut — social post

I built a leakage-aware content-refresh triage model for a FlyRank case study. On 30,000 anonymized content items, I compared an interpretable stale-and-visible rule with logistic regression, a decision tree, and a class-balanced random forest using a client holdout. The forest raised Precision@50 from 0.240 to 0.680, helping prioritize the first review batch; it is a reviewer aid, not proof that refreshing content causes traffic gains. Read the paper and rerun the notebook: https://azizulabedinazmi.is-a.dev/flyrank-paper.html


## Shareable cut — employer-facing summary

I built a leakage-aware content-refresh prioritization pipeline that compares transparent rules with three supervised models and produces a ranked review queue. It used 30,000 rows from FlyRank's anonymized educational dataset, covering 32 pseudonymized clients and trailing-90-day activity with two comparison windows. On a client holdout, the selected random forest reached ROC AUC 0.747 and Precision@50 0.680 versus 0.240 for the rule baseline, showing stronger review prioritization while remaining explicit about the lack of causal lift evidence.
